# CZKB - Import traces (local / Databricks dual-mode)

Load MLflow traces for a single evaluation run and convert them into the same trace-level flat dataframe shape used by the CZKB data-preparation notebook.

This version runs **either**:
- on a Databricks cluster (`RUN_ON_DBX = True`) — uses `spark` / `dbutils` directly, **or**
- on a local machine (`RUN_ON_DBX = False`) — authenticates via the Databricks CLI profile / SDK, talks to the workspace MLflow tracking server and serving endpoints, and (optionally) uses Databricks Connect for Unity Catalog reads/writes.

All trace-derived evaluation fields come from the MLflow experiment traces. The helper English columns (`user_query_en`, `expected_response_en`, `agent_response_en`) are produced by a per-run translation cache. CZ test-set JSON and CZ KB CSVs are optional enrichment inputs: when they are absent, the notebook still runs end-to-end — `categories_list` and the KB-description columns are emitted empty, and only translations populate the English columns.

## Run mode

In [1]:
import os
# Strip env vars the Databricks VS Code extension injects — its loopback OAuth
# broker URL goes stale on extension/VS Code restarts and hangs the kernel.
# Falling through to the .databrickscfg profile is more robust.
for v in ("DATABRICKS_AUTH_TYPE", "DATABRICKS_METADATA_SERVICE_URL",
          "DATABRICKS_HOST", "DATABRICKS_CLUSTER_ID"):
    os.environ.pop(v, None)
os.environ["DATABRICKS_CONFIG_PROFILE"] = "adb-chat"

In [2]:
# Toggle to run on a Databricks cluster vs. locally from your laptop.
RUN_ON_DBX: bool = False

# Local-only: which Databricks CLI profile to use for auth (matches `databricks --profile <name>`).
DBX_PROFILE: str = "adb-chat"

# Local-only: whether to also write the parsed dataframe back to Unity Catalog
# (requires databricks-connect installed and matching the cluster DBR version).
WRITE_TO_UC: bool = True

In [3]:
!databricks auth login --profile adb-chat

]11;?\Profile adb-chat was successfully saved


In [4]:
# !databricks auth login --profile adb-chat   # OK
!databricks auth describe --profile adb-chat   # OK
!databricks current-user me  --profile adb-chat # OK

]11;?\Host: https://adb-3174992876438447.7.azuredatabricks.net
User: sg7cb@s-mxs.net
Authenticated with: databricks-cli
-----
Current configuration:
  ✓ host: https://adb-3174992876438447.7.azuredatabricks.net (from /Users/SG7CB/.databrickscfg config file)
  ✓ account_id: 85cde0e4-68ed-4a99-b884-abaec50d1f90 (from /Users/SG7CB/.databrickscfg config file)
  ✓ workspace_id: 3174992876438447 (from /Users/SG7CB/.databrickscfg config file)
  ✓ profile: adb-chat (from --profile flag)
  ✓ auth_type: databricks-cli (from /Users/SG7CB/.databrickscfg config file)
  ✓ cloud: Azure
  ✓ audience: https://adb-3174992876438447.7.azuredatabricks.net/oidc/v1/token
  ✓ discovery_url: https://adb-3174992876438447.7.azuredatabricks.net/oidc/.well-known/oauth-authorization-server
]11;?\{
  "active":true,
  "displayName":"Cirovic Donev Ita 6028 ED-EXT",
  "emails": [
    {
      "primary":true,
      "type":"work",
      "value":"sg7cb@s-mxs.net"
    }
  ],
  "externalId":"e7491b88-fd90-4b9b-9cec-89e564

## Imports & auth bootstrap

When running locally we point MLflow at the Databricks tracking server using the CLI profile we already verified with `databricks auth describe --profile adb-chat`. No PAT needed.

In [5]:
%load_ext autoreload
%autoreload 2

In [6]:
import os as _os_grpc
_os_grpc.environ.setdefault("GRPC_VERBOSITY", "ERROR")
_os_grpc.environ.setdefault("GRPC_POLL_STRATEGY", "poll")
_os_grpc.environ.setdefault("GLOG_minloglevel", "2")

'2'

In [7]:
import os
import sys
import importlib
from pathlib import Path

# Make the local hg-ds-evals repo importable when running off-cluster.
# (config_nbs.add_local_paths() only adds /Workspace/... paths that exist on Databricks.)
if not globals().get("RUN_ON_DBX", False):
    _repo_root = Path.cwd()
    while _repo_root != _repo_root.parent and not (_repo_root / "hg_ds_evals").is_dir():
        _repo_root = _repo_root.parent
    if (_repo_root / "hg_ds_evals").is_dir() and str(_repo_root) not in sys.path:
        sys.path.insert(0, str(_repo_root))
        print(f'Local repo root added to sys.path: "{_repo_root}"')

import config_nbs
config_nbs.add_local_paths()

import mlflow
import pandas as pd
import hg_ds_evals.preprocessing.skkb_traces as skkb_traces

importlib.reload(skkb_traces)
build_skkb_dataframe_from_mlflow_search_traces = skkb_traces.build_skkb_dataframe_from_mlflow_search_traces

if not RUN_ON_DBX:
    # Make every Databricks SDK / MLflow call use the named CLI profile.
    os.environ["DATABRICKS_CONFIG_PROFILE"] = DBX_PROFILE

    from databricks.sdk import WorkspaceClient
    _w = WorkspaceClient()
    print(f"Authenticated as: {_w.current_user.me().user_name}")
    print(f"Workspace host:   {_w.config.host}")

    # Point MLflow at the Databricks-hosted tracking server.
    mlflow.set_tracking_uri("databricks")
    print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
else:
    print("Running on Databricks cluster — using attached spark/dbutils/MLflow.")


Path "/Workspace/Users/sg7cb@s-mxs.net/hg-ds-evals/" added!
Path "/Workspace/Users/sg7cb@s-mxs.net/hg-ds-evals/experiments/czkb/" added!
Authenticated as: sg7cb@s-mxs.net
Workspace host:   https://adb-3174992876438447.7.azuredatabricks.net
MLflow tracking URI: databricks


## Run / experiment configuration

In [8]:
all_runs:dict = {
    "czkb_jorao":{
        "experiment_id": "471458066277040",
        "run_id": "11831eebd37a445a889d124baec45a32",
        "mlflow_run_name": "online/adhoc/quiet-hawk/score",
    },

    "april_30": {
        "experiment_id": "471458066277040",
        "run_id": "76ae7ceaf4e94423a838ab70c5493c5a",
        "mlflow_run_name": "online/nightly/2026-04-30/score",
    },

    # SK runs kept for reference / quick switch-back.
    "may_1": {
        "experiment_id": "471458066277040",
        "run_id": "db7ae47eee0348f8a3fd9c6dd21b5b3a",
        "mlflow_run_name": "online/nightly/2026-05-01/score",
    },

    "april_24": {
        "experiment_id": "471458066277040",
        "run_id": "3441808bc8db416fbbce3f4878e94d4d",
        "mlflow_run_name": "online_nightly_2026_04_24_infer",
    },
}

In [9]:
RUN_DATE = "czkb_jorao"
EXPERIMENT_ID = all_runs[RUN_DATE]["experiment_id"]
RUN_ID = all_runs[RUN_DATE]["run_id"]
RUN_NAME = all_runs[RUN_DATE]["mlflow_run_name"].replace("/", "_").replace("-", "_")
print(f"Selected run:"
      f"\nExperiment id: {EXPERIMENT_ID}"
      f"\nRun name: {RUN_NAME}"
      f"\nRun id: {RUN_ID}")

MAX_RESULTS = None


# Optional CZ enrichment inputs. If a file is missing, the notebook still
# runs: translations are produced regardless, and the affected enrichment
# column(s) are filled with empty placeholders so the downstream schema
# stays stable. Drop the CZ test-set JSON / KB CSVs into ../input/ once
# they exist; the enrichment cell uses `Path(...).is_file()` so an empty
# string here is treated as "no file" and skipped gracefully.
TEST_SET_JSON_PATH = ""  # e.g. "../input/test_set_GAI_CZ_CZ.json"
KB_CZ_CSV_PATH = "/Users/SG7CB/Developer/ai-orchestrator/apps/evals/data/datasets/knowledge_base/data/KB_HG_CZ_CZ_2026-01-26_15h45.csv"
KB_EN_CSV_PATH = "/Users/SG7CB/Developer/ai-orchestrator/apps/evals/data/datasets/knowledge_base/data/KB_HG_CZ_EN_2026-01-26_15h45.csv"

DBX_CATALOG: str = "prod_aut_chat00_lab_catalog"
DBX_SCHEMA: str = "private_uc0115_aix_db"
DBX_TABLE: str = f"dts_eval_czkb_exp_002_{RUN_NAME}"
print(f"Databricks Unity Catalog target: {DBX_CATALOG}.{DBX_SCHEMA}.{DBX_TABLE}")

DBX_CLUSTER_ID: str = "0424-122149-m97focly"
# Local-only: cluster id used by Databricks Connect (must be a running interactive cluster).

Selected run:
Experiment id: 471458066277040
Run name: online_adhoc_quiet_hawk_score
Run id: 11831eebd37a445a889d124baec45a32
Databricks Unity Catalog target: prod_aut_chat00_lab_catalog.private_uc0115_aix_db.dts_eval_czkb_exp_002_online_adhoc_quiet_hawk_score


## Fetch traces from MLflow

Works identically locally and on a Databricks cluster — the only difference is the tracking URI we set above.

In [10]:
# When True, force a fresh MLflow re-fetch even if local trace caches
# already exist (../input/traces_{RUN_NAME}.pickle for the pandas fetch,
# and ../input/traces_{RUN_NAME}.jsonl for the JSONL fetch). When False
# (default), the fetch cells skip the slow blob-storage download and load
# from the local cache instead.
FORCE_REFETCH_TRACES = True

In [11]:
import logging
from pathlib import Path

# Quiet the noisy "Connection pool is full" warnings emitted while MLflow
# downloads trace artifacts from blob storage in parallel.
logging.getLogger("urllib3.connectionpool").setLevel(logging.ERROR)

# Local cache for the pandas-shaped MLflow fetch. Avoids re-downloading
# trace artifacts from Azure blob storage on every re-execution.
# Set FORCE_REFETCH_TRACES=True in the config cell to overwrite the cache.
traces_pickle_path = Path(f"../input/traces_{RUN_NAME}.pickle")

if traces_pickle_path.is_file() and not FORCE_REFETCH_TRACES:
    print(f"loading cached pandas traces from {traces_pickle_path}")
    print("(set FORCE_REFETCH_TRACES=True in the config cell to force a fresh MLflow fetch)")
    traces_df = pd.read_pickle(traces_pickle_path)
else:
    if traces_pickle_path.is_file():
        print(f"FORCE_REFETCH_TRACES=True — overwriting cache at {traces_pickle_path}")
    else:
        print(f"no cache at {traces_pickle_path} — fetching from MLflow")
    traces_df = mlflow.search_traces(
        locations=[EXPERIMENT_ID],   # `experiment_ids` is deprecated in MLflow 3.x
        run_id=RUN_ID,
        max_results=MAX_RESULTS,
        order_by=["timestamp_ms DESC"],
    )
    traces_pickle_path.parent.mkdir(parents=True, exist_ok=True)
    traces_df.to_pickle(traces_pickle_path)
    print(f"saved cache -> {traces_pickle_path}")

print(f"Shape: {traces_df.shape}")
print(f"Columns: {traces_df.columns.tolist()}")
traces_df.head()

FORCE_REFETCH_TRACES=True — overwriting cache at ../input/traces_online_adhoc_quiet_hawk_score.pickle
saved cache -> ../input/traces_online_adhoc_quiet_hawk_score.pickle
Shape: (1123, 12)
Columns: ['trace_id', 'trace', 'client_request_id', 'state', 'request_time', 'execution_duration', 'request', 'response', 'trace_metadata', 'tags', 'spans', 'assessments']


,trace_id,trace,client_request_id,state,request_time,execution_duration,request,response,trace_metadata,tags,spans,assessments
0,tr-52cec032a20617f1df897c75aba3c590,"{""info"": {""trace_id"": ""tr-52cec032a20617f1df89...",tr-52cec032a20617f1df897c75aba3c590,OK,1778600684399,15027,None,None,"{'mlflow.trace.sizeStats': '{""total_size_bytes...","{'eval.test_case_id': 'Test case 777', 'eval.k...","[{'trace_id': 'Us7AMqIGF/HfiXx1q6PFkA==', 'spa...",[{'assessment_id': 'a-54517f056860477abe8eafe7...
1,tr-58e739b97eafab5a3e5585c986d35aed,"{""info"": {""trace_id"": ""tr-58e739b97eafab5a3e55...",tr-58e739b97eafab5a3e5585c986d35aed,OK,1778600684397,17471,None,None,"{'mlflow.trace.sizeStats': '{""total_size_bytes...","{'eval.test_case_id': 'Test case 776', 'eval.k...","[{'trace_id': 'WOc5uX6vq1o+VYXJhtNa7Q==', 'spa...",[{'assessment_id': 'a-b0de132a63d6401fafa24555...
2,tr-3b0927ebab2ab10f3731459e27cf4ed0,"{""info"": {""trace_id"": ""tr-3b0927ebab2ab10f3731...",tr-3b0927ebab2ab10f3731459e27cf4ed0,OK,1778600684393,18539,None,None,"{'mlflow.trace.sizeStats': '{""total_size_bytes...","{'eval.test_case_id': 'Test case 771', 'eval.k...","[{'trace_id': 'Owkn66sqsQ83MUWeJ89O0A==', 'spa...",[{'assessment_id': 'a-9c936f4d4bbd43058dacc6e8...
3,tr-6a4292d6de86f2ac883e65219e4597e4,"{""info"": {""trace_id"": ""tr-6a4292d6de86f2ac883e...",tr-6a4292d6de86f2ac883e65219e4597e4,OK,1778600614287,26809,None,None,"{'mlflow.trace.sizeStats': '{""total_size_bytes...","{'eval.test_case_id': 'Test case 770', 'eval.k...","[{'trace_id': 'akKS1t6G8qyIPmUhnkWX5A==', 'spa...",[{'assessment_id': 'a-cdedd1082a8e43e381ce3f2e...
4,tr-1e8c93b73433755edd7ee64842b3cfc2,"{""info"": {""trace_id"": ""tr-1e8c93b73433755edd7e...",tr-1e8c93b73433755edd7ee64842b3cfc2,OK,1778600614285,15891,None,None,"{'mlflow.trace.sizeStats': '{""total_size_bytes...","{'eval.test_case_id': 'Test case 769', 'eval.k...","[{'trace_id': 'HoyTtzQzdV7dfuZIQrPPwg==', 'spa...",[{'assessment_id': 'a-7070c072d3a7498ba3174cb6...


In [12]:
#traces_df.iloc[1]["trace_metadata"]
#traces_df.iloc[1]["tags"]
#traces_df.iloc[1]["spans"]
#traces_df.iloc[1]["assessments"]

## Parse traces

In [13]:
parse_result = build_skkb_dataframe_from_mlflow_search_traces(traces_df)

trace_level_df = parse_result.dataframe
parse_errors = parse_result.parse_errors
unmapped_trace_ids = parse_result.unmapped_trace_ids

print(f"Run: {RUN_NAME}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Trace rows fetched: {len(traces_df):,}")
print(f"Parsed rows: {len(trace_level_df):,}")
print(f"Parse errors: {len(parse_errors):,}")
print(f"Rows using trace_id as test_case_id fallback: {len(unmapped_trace_ids):,}")

if parse_errors:
    print(parse_errors[:5])

Run: online_adhoc_quiet_hawk_score
Tracking URI: databricks
Trace rows fetched: 1,123
Parsed rows: 1,123
Parse errors: 0
Rows using trace_id as test_case_id fallback: 0


### `agent_response` coverage check

Catches the case where this notebook is pointed at a run that
pre-dates the `agent_answer` AGENT-span instrumentation
(GCAI-2566 / GCAI-3581). On those runs the strict extractor
returns `""` for every trace, which is unscoreable — better to
fail loud here than to ship empty answers to downstream judges.

- **100% empty** → raise (wrong run picked / instrumentation regression).
- **> warn threshold empty** → print a warning; investigate but continue.
- Otherwise print coverage and proceed.

In [ ]:
AGENT_RESPONSE_EMPTY_WARN_FRACTION = 0.10

_n_total = len(trace_level_df)
_n_empty = int((trace_level_df["agent_response"].fillna("").str.len() == 0).sum())
_empty_fraction = _n_empty / _n_total if _n_total else 0.0

print(f"agent_response coverage: {_n_total - _n_empty}/{_n_total} "
      f"({100 * (1 - _empty_fraction):.1f}%)")
print(f"agent_response empty:    {_n_empty}/{_n_total} ({100 * _empty_fraction:.1f}%)")

if _n_total and _n_empty == _n_total:
    raise RuntimeError(
        f"agent_response is empty for ALL {_n_total} traces in {RUN_NAME!r}. "
        "The strict extractor reads only the `agent_answer` AGENT span (post "
        "GCAI-2566 / GCAI-3581). Either this run pre-dates that instrumentation "
        "or the orchestrator stopped emitting it \u2014 either way, this dataset is "
        "not scoreable on response quality. Pick a newer run or check the orchestrator."
    )
elif _empty_fraction > AGENT_RESPONSE_EMPTY_WARN_FRACTION:
    print(
        f"WARNING: {100 * _empty_fraction:.1f}% of traces have empty agent_response "
        f"(threshold {100 * AGENT_RESPONSE_EMPTY_WARN_FRACTION:.0f}%). "
        "Investigate whether the `agent_answer` span is being emitted reliably "
        "before scoring these traces."
    )

In [12]:
print(trace_level_df.iloc[0])

test_case_id                                                                   Test case 777
trace_id                                                 tr-52cec032a20617f1df897c75aba3c590
request_time                                                                   1778600684399
execution_duration_ms                                                                  15027
user_query                                  İnanç için 10.000 kron transfer ücreti ne kadar?
knowledge_search_run_count                                                                 1
knowledge_search_final_run_index                                                           1
knowledge_search_runs                      [{'run_index': 1, 'knowledge_search_span_id': ...
reranked_kb_context                        [KB_HG_CZ_CZ_2026-01-26_15h45]\nSTART_TRANSFER...
kb_version                                                      KB_HG_CZ_CZ_2026-01-26_15h45
reranked_enum_ids                          [START_TRANSFER@FOREIGN, GI

In [13]:
# JSONL traces — separate MLflow fetch with return_type="list" (yields
# Trace objects, not a pandas DataFrame). Cached at
# ../input/traces_{RUN_NAME}.jsonl; skipped on re-run unless
# FORCE_REFETCH_TRACES=True.
traces_jsonl_path = Path(f"../input/traces_{RUN_NAME}.jsonl")

if traces_jsonl_path.is_file() and not FORCE_REFETCH_TRACES:
    print(f"JSONL cache already at {traces_jsonl_path} — skipping fetch")
    print("(set FORCE_REFETCH_TRACES=True in the config cell to force a fresh MLflow fetch)")
else:
    if traces_jsonl_path.is_file():
        print(f"FORCE_REFETCH_TRACES=True — overwriting {traces_jsonl_path}")
    traces = mlflow.search_traces(
        locations=[EXPERIMENT_ID],
        run_id=RUN_ID,
        max_results=MAX_RESULTS,
        order_by=["timestamp_ms DESC"],
        return_type="list",   # default is "pandas"
    )
    traces_jsonl_path.parent.mkdir(parents=True, exist_ok=True)
    with traces_jsonl_path.open("w") as f:
        for t in traces:
            f.write(t.to_json() + "\n")
    print(f"saved {len(traces)} traces -> {traces_jsonl_path}")

FORCE_REFETCH_TRACES=True — overwriting ../input/traces_online_adhoc_quiet_hawk_score.jsonl


Python(52463) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(52469) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(52484) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(52491) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(52492) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(52493) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(52494) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(52495) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(52496) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(52497) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(52498) Malloc

saved 1123 traces -> ../input/traces_online_adhoc_quiet_hawk_score.jsonl


## Add enrichment columns and persistent English text caches

Get the host + token for the translation HTTP calls — locally we use the Databricks SDK instead of `spark.conf` / `dbutils`.

In [14]:
import concurrent.futures
import json
import time
from pathlib import Path
from urllib import error, request

TRANSLATE_MODEL = "gpt-5-1"
TRANSLATE_MAX_CONCURRENCY = 20
TRANSLATE_MAX_OUTPUT_TOKENS = 1024
TRANSLATE_RETRY_ATTEMPTS = 4  # total attempts per call (1 initial + 3 retries on transient errors)

TRANSLATE_SYSTEM_PROMPT = (
    "You translate text into English. Translate the user message "
    "correctly and concisely. Preserve banking terminology, product "
    "names, numbers, and formatting. If the input is already in "
    "English, output it unchanged. Do NOT add commentary, quotes, or "
    "explanations. Output ONLY the English text."
)

# Per-run translation cache: CZ->EN for user_query / expected_response /
# agent_response. Lives next to the input data, named after the run so it
# stays connected to that specific MLflow run. On re-import of the same run
# it is a pure cache hit; for a new run it is built lazily.
TRANSLATIONS_CACHE_PATH = f"../input/czkb_translations_{RUN_NAME}.json"

# Resolve workspace host + bearer token in a way that works both on the cluster and locally.
if RUN_ON_DBX:
    _dbx_host = spark.conf.get("spark.databricks.workspaceUrl")  # noqa: F821 (provided by DBR)
    _dbx_token = (
        dbutils.notebook.entry_point.getDbutils()  # noqa: F821 (provided by DBR)
        .notebook()
        .getContext()
        .apiToken()
        .get()
    )
else:
    from databricks.sdk import WorkspaceClient
    _w_local = WorkspaceClient()
    _dbx_host = _w_local.config.host.replace("https://", "").rstrip("/")
    _dbx_token = _w_local.config.authenticate()["Authorization"].removeprefix("Bearer ")

_translate_url = f"https://{_dbx_host}/serving-endpoints/chat/completions"


def _atomic_write_json(path: str, payload: dict) -> None:
    target_path = Path(path)
    target_path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = target_path.with_suffix(target_path.suffix + ".tmp")
    with temp_path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2, sort_keys=True)
    temp_path.replace(target_path)


def _load_json(path: str, default: dict) -> dict:
    try:
        with open(path, "r", encoding="utf-8") as handle:
            payload = json.load(handle)
    except FileNotFoundError:
        return default
    return payload if isinstance(payload, dict) else default


def _translate_one(text: str) -> str:
    """Single translation call with retry on transient upstream errors.

    Retries on HTTP 5xx (transient OpenAI-via-Databricks 500s, gateway
    timeouts, etc.) and on network/socket errors with exponential backoff
    (1s, 2s, 4s). 4xx errors are NOT retried — those reflect a bad request.
    """
    payload = {
        "model": TRANSLATE_MODEL,
        "messages": [
            {"role": "system", "content": TRANSLATE_SYSTEM_PROMPT},
            {"role": "user", "content": text},
        ],
        "max_tokens": TRANSLATE_MAX_OUTPUT_TOKENS,
    }
    encoded = json.dumps(payload).encode("utf-8")
    for attempt in range(TRANSLATE_RETRY_ATTEMPTS):
        req = request.Request(
            _translate_url,
            data=encoded,
            headers={
                "Authorization": f"Bearer {_dbx_token}",
                "Content-Type": "application/json",
            },
            method="POST",
        )
        try:
            with request.urlopen(req, timeout=120) as response:
                response_payload = json.loads(response.read().decode("utf-8"))
            choice = ((response_payload.get("choices") or [{}])[0].get("message") or {})
            return (choice.get("content") or "").strip()
        except error.HTTPError as exc:
            detail = exc.read().decode("utf-8", errors="ignore")
            transient = 500 <= exc.code < 600
            if transient and attempt < TRANSLATE_RETRY_ATTEMPTS - 1:
                time.sleep(2 ** attempt)
                continue
            raise RuntimeError(
                f"Translation request failed for text {text[:80]!r} "
                f"(status {exc.code}, attempt {attempt + 1}/{TRANSLATE_RETRY_ATTEMPTS}): {detail}"
            ) from exc
        except (error.URLError, TimeoutError, OSError) as exc:
            if attempt < TRANSLATE_RETRY_ATTEMPTS - 1:
                time.sleep(2 ** attempt)
                continue
            raise RuntimeError(
                f"Translation network error for text {text[:80]!r} "
                f"(attempt {attempt + 1}/{TRANSLATE_RETRY_ATTEMPTS}): {exc}"
            ) from exc


def _translate_unique_texts(texts: list[str]) -> tuple[dict[str, str], list[tuple[str, str]]]:
    """Translate texts in parallel.

    Returns (successes, failures). Individual failures do NOT abort the
    batch — they are collected separately so the successful translations
    are preserved and can be persisted by the caller.
    """
    unique_texts = sorted({text for text in texts if isinstance(text, str) and text.strip()})
    if not unique_texts:
        return {}, []
    max_workers = min(TRANSLATE_MAX_CONCURRENCY, len(unique_texts))
    successes: dict[str, str] = {}
    failures: list[tuple[str, str]] = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_text = {executor.submit(_translate_one, t): t for t in unique_texts}
        for fut in concurrent.futures.as_completed(future_to_text):
            text = future_to_text[fut]
            try:
                successes[text] = fut.result()
            except Exception as exc:
                failures.append((text, str(exc)))
    return successes, failures


def _attach_kb(ids, lang_table: dict[str, dict[str, str]]) -> list[dict[str, object]]:
    attached = []
    for enum_id in ids if isinstance(ids, list) else []:
        enum_key = str(enum_id)
        kb_row = lang_table.get(enum_key)
        attached.append({
            "enum_id": enum_key,
            "description": (kb_row or {}).get("kb.description", ""),
            "missing": kb_row is None,
        })
    return attached


# -- Test-set JSON: optional. When present, used ONLY for test_case_id
#    resolution + categories_list. When absent, categories_list is left empty
#    and test_case_id falls back to the trace_id assigned upstream.
test_set_path_exists = Path(TEST_SET_JSON_PATH).is_file()
if test_set_path_exists:
    test_set_payload = _load_json(TEST_SET_JSON_PATH, {})
else:
    print(f"test-set JSON not found at {TEST_SET_JSON_PATH} — categories_list will be empty")
    test_set_payload = {}

user_query_to_test_case_id = {}
for test_case_id, record in test_set_payload.items():
    if not isinstance(record, dict):
        continue
    user_query = record.get("user_query")
    if isinstance(user_query, str) and user_query.strip() and user_query not in user_query_to_test_case_id:
        user_query_to_test_case_id[user_query] = test_case_id


def _resolve_test_case_id(row) -> str | None:
    test_case_id = row.get("test_case_id")
    if isinstance(test_case_id, str) and test_case_id in test_set_payload:
        return test_case_id
    user_query = row.get("user_query")
    if isinstance(user_query, str):
        return user_query_to_test_case_id.get(user_query)
    return None


if test_set_payload:
    resolved_test_case_ids = trace_level_df.apply(_resolve_test_case_id, axis=1)
    matched_test_set_rows = int(resolved_test_case_ids.notna().sum())
    print(f"test-set matches: {matched_test_set_rows}/{len(trace_level_df)}")
    trace_level_df["categories_list"] = resolved_test_case_ids.apply(
        lambda test_case_id: ((test_set_payload.get(test_case_id) or {}).get("categories_list") or []) if isinstance(test_case_id, str) else []
    )
else:
    trace_level_df["categories_list"] = [[] for _ in range(len(trace_level_df))]

# -- KB descriptions (CZ + EN): optional. When the CSVs are absent, the
#    kb-description columns are still emitted (with empty descriptions and
#    missing=True) so the downstream schema stays stable.
reranked_enum_values = trace_level_df["reranked_enum_ids"].tolist() if "reranked_enum_ids" in trace_level_df.columns else [[] for _ in range(len(trace_level_df))]
post_prune_enum_values = trace_level_df["post_prune_enum_ids"].tolist() if "post_prune_enum_ids" in trace_level_df.columns else [[] for _ in range(len(trace_level_df))]

kb_csv_paths_exist = Path(KB_CZ_CSV_PATH).is_file() and Path(KB_EN_CSV_PATH).is_file()
if kb_csv_paths_exist:
    kb_cz = pd.read_csv(KB_CZ_CSV_PATH, sep="|", dtype=str).fillna("")
    kb_en = pd.read_csv(KB_EN_CSV_PATH, sep="|", dtype=str).fillna("")
    cz_by_id = kb_cz.set_index("kb.knowledgeId").to_dict(orient="index")
    en_by_id = kb_en.set_index("kb.knowledgeId").to_dict(orient="index")
else:
    print(f"KB CSVs not found at {KB_CZ_CSV_PATH} / {KB_EN_CSV_PATH} — kb description columns will be empty")
    cz_by_id, en_by_id = {}, {}

trace_level_df["reranked_enums_kb_cz"] = [_attach_kb(ids, cz_by_id) for ids in reranked_enum_values]
trace_level_df["reranked_enums_kb_en"] = [_attach_kb(ids, en_by_id) for ids in reranked_enum_values]
trace_level_df["post_prune_candidates_kb_cz"] = [_attach_kb(ids, cz_by_id) for ids in post_prune_enum_values]
trace_level_df["post_prune_candidates_kb_en"] = [_attach_kb(ids, en_by_id) for ids in post_prune_enum_values]

# -- Translations: load per-run cache, translate misses, save AFTER EACH FIELD.
#    Per-item failures do NOT abort the run — they are logged and re-attempted
#    on the next execution because they remain absent from the cache.
print(f"\ntranslation cache: {TRANSLATIONS_CACHE_PATH}")
translations_cache = _load_json(
    TRANSLATIONS_CACHE_PATH,
    {"user_query": {}, "expected_response": {}, "agent_response": {}},
)
for section in ("user_query", "expected_response", "agent_response"):
    translations_cache.setdefault(section, {})


def _translate_field(field_name: str, source_values: list[str], cache: dict) -> tuple[list[str], bool]:
    section = cache[field_name]
    unique_inputs = sorted({t for t in source_values if isinstance(t, str) and t.strip()})
    missing = [t for t in unique_inputs if t not in section]
    print(f"  {field_name:>18s}_en: cached {len(unique_inputs) - len(missing)}/{len(unique_inputs)}"
          f", to translate {len(missing)}")
    if missing:
        successes, failures = _translate_unique_texts(missing)
        section.update(successes)
        # Persist after every field so partial progress survives later failures.
        _atomic_write_json(TRANSLATIONS_CACHE_PATH, cache)
        if failures:
            print(f"    WARN: {len(failures)} translations failed (will retry on next run). Examples:")
            for text, err in failures[:3]:
                print(f"      - {text[:60]!r}: {err[:120]}")
    out = [section.get(t, "") if isinstance(t, str) and t.strip() else "" for t in source_values]
    return out, bool(missing)


user_query_values = trace_level_df["user_query"].tolist() if "user_query" in trace_level_df.columns else [""] * len(trace_level_df)
expected_response_values = trace_level_df["expected_response"].tolist() if "expected_response" in trace_level_df.columns else [""] * len(trace_level_df)
agent_response_values = trace_level_df["agent_response"].tolist() if "agent_response" in trace_level_df.columns else [""] * len(trace_level_df)

dirty = False
for field, values, target_col in (
    ("user_query",         user_query_values,         "user_query_en"),
    ("expected_response",  expected_response_values,  "expected_response_en"),
    ("agent_response",     agent_response_values,     "agent_response_en"),
):
    translated, did_translate = _translate_field(field, values, translations_cache)
    trace_level_df[target_col] = translated
    dirty = dirty or did_translate

if dirty:
    print(f"saved translations cache -> {TRANSLATIONS_CACHE_PATH}")
else:
    print("translations cache unchanged (all hits cached).")

print(f"\nuser_query_en non-empty:        {(trace_level_df['user_query_en'].str.len() > 0).sum()}/{len(trace_level_df)}")
print(f"expected_response_en non-empty: {(trace_level_df['expected_response_en'].str.len() > 0).sum()}/{len(trace_level_df)}")
print(f"agent_response_en non-empty:    {(trace_level_df['agent_response_en'].str.len() > 0).sum()}/{len(trace_level_df)}")
print(f"categories_list populated:      {sum(bool(v) for v in trace_level_df['categories_list'])}/{len(trace_level_df)}")
print(f"reranked_enums_kb_cz populated: {sum(bool(v) for v in trace_level_df['reranked_enums_kb_cz'])}/{len(trace_level_df)}")
print(f"post_prune_candidates_kb_cz populated: {sum(bool(v) for v in trace_level_df['post_prune_candidates_kb_cz'])}/{len(trace_level_df)}")


Python(9281) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


test-set JSON not found at  — categories_list will be empty

translation cache: ../input/czkb_translations_online_adhoc_quiet_hawk_score.json
          user_query_en: cached 1123/1123, to translate 0
   expected_response_en: cached 1123/1123, to translate 0
      agent_response_en: cached 1123/1123, to translate 0
translations cache unchanged (all hits cached).

user_query_en non-empty:        1123/1123
expected_response_en non-empty: 1123/1123
agent_response_en non-empty:    1123/1123
categories_list populated:      0/1123
reranked_enums_kb_cz populated: 1092/1123
post_prune_candidates_kb_cz populated: 1093/1123


In [15]:
trace_level_df.iloc[1][["user_query", "user_query_en", "agent_response", "agent_response_en", "expected_response", "expected_response_en"]]

user_query                                     What is the exchange rate?
user_query_en                                  What is the exchange rate?
agent_response          I can’t see a specific rate directly here.\n\n...
agent_response_en       I can’t see a specific rate directly here.\n\n...
expected_response       Krystýno, which currencies would you like to c...
expected_response_en    Krystýno, which currencies would you like to c...
Name: 1, dtype: object

In [16]:
if not RUN_ON_DBX:
    Path("traces").mkdir(exist_ok=True)
    trace_level_df.to_csv(f"traces/czkb_traces_enriched_{RUN_NAME}.csv", index=False)

## View data

In [ ]:
# # `display(...)` is a Databricks-only helper; fall back to a plain dataframe locally.
# if RUN_ON_DBX:
#     display(trace_level_df)  # noqa: F821 (provided by DBR)
# else:
#     trace_level_df

In [ ]:
trace_level_df.columns

In [21]:
trace_level_df["last_agent"].value_counts(dropna=False)

last_agent
daily_banking_agent    1102
main_agent               21
Name: count, dtype: int64

In [22]:
cols = ["test_case_id",
        "user_query", "user_query_en",
        "raw_vector_db_retrieved_enum_count", "pre_prune_enum_count", "post_prune_enum_count", "reranked_enum_ids", "expected_enums",
        "reranker_raw_selected_ids", "reranker_valid_selected_ids", "reranker_invalid_selected_ids", "reranker_unselected_context_ids", "reranker_selection_status", "reranker_selection_violations"]
missing_cols = [column for column in cols if column not in trace_level_df.columns]
if missing_cols:
    print(f"Skipping missing preview columns: {missing_cols}")
cols = [column for column in cols if column in trace_level_df.columns]
trace_level_df[cols].sort_values(by="test_case_id", ascending=True).head()

,test_case_id,user_query,user_query_en,raw_vector_db_retrieved_enum_count,pre_prune_enum_count,post_prune_enum_count,reranked_enum_ids,expected_enums,reranker_raw_selected_ids,reranker_valid_selected_ids,reranker_invalid_selected_ids,reranker_unselected_context_ids,reranker_selection_status,reranker_selection_violations
720,Test case 0,Proč mi volali před chvílí?,Why did they call me a moment ago?,0,0,0,[],"[CALL_PHONE_AUTHORISED_V2, GEORGE@BANKER_CONFIRM]",[],[],[],[],not_applicable,[]
719,Test case 1,Volali jste mi?,Did you call me?,0,0,0,[],"[CALL_PHONE_AUTHORISED_V2, GEORGE@BANKER_CONFIRM]",[],[],[],[],not_applicable,[]
913,Test case 1000,"Hi, I want to open a savings account.","Hi, I want to open a savings account.",180,75,30,"[SAVING@APPLICATION, SAVING@ABOUT, SAVING@FEES...","[SAVING@APPLICATION, SAVING@FEES, SAVING@INTER...","[""SAVING@APPLICATION"", ""SAVING@ABOUT"", ""SAVING...","[""SAVING@APPLICATION"", ""SAVING@ABOUT"", ""SAVING...",[],[],ok,[]
912,Test case 1001,How do I open a savings account?,How do I open a savings account?,180,86,31,[],"[SAVING@APPLICATION, SAVING@FEES, SAVING@INTER...",[],[],[],[],missing_selected_ids,"[""reranker_selected_ids_missing""]"
911,Test case 1002,I need a savings account.,I need a savings account.,0,0,0,[],"[SAVING@APPLICATION, SAVING@FEES, SAVING@INTER...",[],[],[],[],missing_rerank,"[""reranker_span_missing""]"


In [23]:
for enum_column in ("post_prune_enum_ids", "pre_prune_enum_ids"):
    if enum_column in trace_level_df.columns:
        print(f"empty {enum_column}:", sum(trace_level_df[enum_column].apply(lambda x: x == [])))
    else:
        print(f"{enum_column} missing")
count_cols = [
    "knowledge_search_run_count",
    "raw_vector_db_query_count",
    "raw_vector_db_retrieved_enum_count",
    "pre_prune_enum_count",
    "post_prune_enum_count",
    "prune_candidates_in",
    "prune_candidates_out",
    "prune_candidates_dropped",
]
count_cols = [column for column in count_cols if column in trace_level_df.columns]
if count_cols:
    trace_level_df[["test_case_id", *count_cols]].describe()

empty post_prune_enum_ids: 30
empty pre_prune_enum_ids: 30


### Execution time stats

In [24]:
trace_level_df["execution_duration_s"] = trace_level_df["execution_duration_ms"] / 1000
trace_level_df["execution_duration_min"] = trace_level_df["execution_duration_ms"] / (1000 * 60)

trace_level_df["execution_duration_s"].describe()

count    1123.000000
mean       23.354262
std        14.902453
min         5.395000
25%        16.825500
50%        20.809000
75%        26.054500
max       333.004000
Name: execution_duration_s, dtype: float64

## Save parsed traces to Unity Catalog (optional)

Local execution requires `databricks-connect==<your-DBR-version>`. Set `WRITE_TO_UC = True` (and `RUN_ON_DBX = False` for the local path) to enable.

In [25]:
del traces_df; 
import gc; gc.collect()

238

In [26]:
# Write the table to UC
if not WRITE_TO_UC:
    print("WRITE_TO_UC=False — skipping Unity Catalog write.")
elif trace_level_df.empty:
    raise ValueError("trace_level_df is empty; nothing to write")
else:
    trace_level_to_write = trace_level_df.copy()

    json_string_columns = [
        "reranked_enum_ids",
        "expected_enums",
        "knowledge_search_runs",
        "raw_vector_db_retrieved_enum_ids",
        "raw_vector_db_retrieved_count_by_query",
        "raw_vector_db_query_limits",
        "pre_prune_enum_ids",
        "post_prune_enum_ids",
        "categories_list",
        "reranked_enums_kb_cz",
        "reranked_enums_kb_en",
        "post_prune_candidates_kb_cz",
        "post_prune_candidates_kb_en",
        "agents_called",
        "tools_called",
        "trace_invariant_violations",
    ]

    for column in json_string_columns:
        if column in trace_level_to_write.columns:
            trace_level_to_write[column] = trace_level_to_write[column].apply(
                lambda value: json.dumps(value, ensure_ascii=False)
                if isinstance(value, (list, dict))
                else (value if isinstance(value, str) else "[]")
            )

    # Force a stable nullable-float dtype for assessment-derived numerics. Without
    # this, a run that skipped a scorer (e.g. all-None expert_score) lands as
    # object dtype → Spark NullType → column dropped under overwriteSchema=true
    # → any UC row filter / column mask referencing it then fails on read.
    nullable_float_cols = ("expert_score", "enum_relevance_score")
    for column in nullable_float_cols:
        if column in trace_level_to_write.columns:
            trace_level_to_write[column] = (
                pd.to_numeric(trace_level_to_write[column], errors="coerce").astype("Float64")
            )

    # Fail fast if a column the schema/policy depends on ever goes missing.
    required_cols = {"test_case_id", "trace_id", "expert_score", "expert_rationale", "enum_relevance_score"}
    missing = required_cols - set(trace_level_to_write.columns)
    if missing:
        raise ValueError(f"refusing to write — missing required columns: {sorted(missing)}")

    if RUN_ON_DBX:
        spark_session = spark  # noqa: F821 (provided by DBR)
    else:
        # Requires `databricks-connect` matching the cluster's DBR version.
        from databricks.connect import DatabricksSession
        spark_session = (
            DatabricksSession.builder
            .profile(DBX_PROFILE)
            .clusterId(DBX_CLUSTER_ID)
            .getOrCreate()
        )

    sdf = spark_session.createDataFrame(trace_level_to_write)
    row_count = len(trace_level_to_write)

    full_table = f"{DBX_CATALOG}.{DBX_SCHEMA}.{DBX_TABLE}"
    (
        sdf
        .write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table)
    )

    print(f"wrote {row_count} rows to {full_table}")

Python(83024) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(91281) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(91285) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


wrote 1123 rows to prod_aut_chat00_lab_catalog.private_uc0115_aix_db.dts_eval_czkb_exp_002_online_adhoc_quiet_hawk_score


In [27]:
# Check that the table was written correctly by reading it back and comparing row counts.
if WRITE_TO_UC:
    if RUN_ON_DBX:
        spark_session = spark  # noqa: F821 (provided by DBR)
    else:
        from databricks.connect import DatabricksSession
        spark_session = (
            DatabricksSession.builder
            .profile(DBX_PROFILE)
            .clusterId(DBX_CLUSTER_ID)
            .getOrCreate()
        )

    full_table = f"{DBX_CATALOG}.{DBX_SCHEMA}.{DBX_TABLE}"
    read_back_df = spark_session.table(full_table)
    read_back_count = read_back_df.count()
    original_count = len(trace_level_df)
    if read_back_count != original_count:
        raise ValueError(f"Row count mismatch: wrote {original_count} rows but read back {read_back_count} rows from {full_table}")
    else:
        print(f"Row count verified: {read_back_count} rows in {full_table}")

Python(92291) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Row count verified: 1123 rows in prod_aut_chat00_lab_catalog.private_uc0115_aix_db.dts_eval_czkb_exp_002_online_adhoc_quiet_hawk_score


In [17]:
# ──────────────────────────────────────────────────────────────────────────────
# Patch: push refreshed KB-description columns into the 002 checkpoint
# ──────────────────────────────────────────────────────────────────────────────
# Why this cell exists:
#   Re-running the import above refreshed `trace_level_df` with proper
#   reranked_enums_kb_cz / _en (now that KB_CZ_CSV_PATH and KB_EN_CSV_PATH
#   are set). The existing checkpoint at
#       checkpoints/evals_czkb_exp_002_…_score.csv
#   still carries the stale "missing=true" placeholders from when the
#   eval first ran. The judge scored against `reranked_kb_context`
#   (always populated), so the scores themselves are sound — we only
#   need to update the two report-side description columns.
#
# What it does:
#   1. Save a timestamped .bak of the checkpoint first.
#   2. Build a {trace_id -> new_cz_json, new_en_json} map from
#      trace_level_df.
#   3. Overwrite ONLY reranked_enums_kb_cz / _en in the checkpoint.
#   4. Refuse to save if the refreshed source is itself still empty
#      (defensive — catches a forgotten KB_*_CSV_PATH setting).
#
# Re-run order: (1) set KB_CZ_CSV_PATH / KB_EN_CSV_PATH, (2) re-run the
# import cells above so trace_level_df is rebuilt, (3) run this cell.

from datetime import datetime
import shutil
import pandas as pd
import json

PATCH_TARGET_CHECKPOINT = Path(
    "checkpoints/"
    "evals_czkb_exp_002_baseline_no_expected_enums_high_online_adhoc_quiet_hawk_score.csv"
)
JOIN_KEY_PREFERENCE = ("trace_id", "test_case_id")
COLUMNS_TO_PATCH    = ("reranked_enums_kb_cz", "reranked_enums_kb_en")

# ── 0. preflight ─────────────────────────────────────────────────────────────
if not PATCH_TARGET_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Checkpoint not found: {PATCH_TARGET_CHECKPOINT}")
try:
    _ = trace_level_df
except NameError as e:
    raise RuntimeError(
        "trace_level_df not in scope — re-run the import cells above "
        "before running this patch cell."
    ) from e
for c in COLUMNS_TO_PATCH:
    if c not in trace_level_df.columns:
        raise RuntimeError(f"trace_level_df is missing column {c!r}")

# ── 1. timestamped backup of the checkpoint ──────────────────────────────────
backup_path = PATCH_TARGET_CHECKPOINT.with_name(
    f"{PATCH_TARGET_CHECKPOINT.name}.bak.{datetime.now().strftime('%Y%m%d_%H%M%S')}"
)
shutil.copy2(PATCH_TARGET_CHECKPOINT, backup_path)
print(f"backup written:\n  {backup_path}")

# ── 2. load checkpoint + pick a join key present on both sides ───────────────
ckp_df = pd.read_csv(PATCH_TARGET_CHECKPOINT, dtype=str).fillna("")
print(f"\ncheckpoint rows: {len(ckp_df):>5}")
print(f"source rows:     {len(trace_level_df):>5}")

join_key = next(
    (k for k in JOIN_KEY_PREFERENCE
     if k in ckp_df.columns and k in trace_level_df.columns),
    None,
)
if join_key is None:
    raise RuntimeError(
        f"None of {JOIN_KEY_PREFERENCE!r} present on both sides — "
        "cannot align rows."
    )
print(f"join key:        {join_key!r}")

# ── 3. defensive: confirm refreshed source actually carries descriptions ─────
def _has_any_desc(value) -> bool:
    items = value if isinstance(value, list) else None
    if items is None:
        try:
            items = json.loads(value) if isinstance(value, str) and value else []
        except json.JSONDecodeError:
            return False
    return isinstance(items, list) and any(
        isinstance(x, dict) and str(x.get("description", "")).strip()
        for x in items
    )

src_has_desc = int(trace_level_df["reranked_enums_kb_cz"].apply(_has_any_desc).sum())
print(f"\nsource rows with ≥1 non-empty CZ description: "
      f"{src_has_desc} / {len(trace_level_df)}")
if src_has_desc == 0:
    raise RuntimeError(
        "Refreshed source has ZERO descriptions — re-importing did NOT "
        "fix the data. Check KB_CZ_CSV_PATH / KB_EN_CSV_PATH point at "
        "real files and re-run the import cells above before re-running "
        "this patch cell. Checkpoint left untouched."
    )

# ── 4. build per-key maps from source, then overwrite checkpoint columns ────
def _to_json_str(v) -> str:
    """trace_level_df cells are usually Python lists right after import;
    serialise to JSON so the checkpoint stays str-typed."""
    if isinstance(v, list):
        return json.dumps(v, ensure_ascii=False)
    if isinstance(v, str):
        return v
    return ""

src_keys = trace_level_df[join_key].astype(str)
ckp_keys = ckp_df[join_key].astype(str)

n_updated = {}
for col in COLUMNS_TO_PATCH:
    src_map = dict(zip(src_keys, trace_level_df[col].apply(_to_json_str)))
    new_values = ckp_keys.map(src_map)
    # Keep the existing value for any checkpoint row that doesn't appear
    # in the refreshed source — never silently blank out a column.
    ckp_df[col] = new_values.fillna(ckp_df[col])
    n_updated[col] = int(new_values.notna().sum())

print("\nrows updated per column:")
for col, n in n_updated.items():
    print(f"  {col:<25}  {n} / {len(ckp_df)}")

# ── 5. save in place ────────────────────────────────────────────────────────
ckp_df.to_csv(PATCH_TARGET_CHECKPOINT, index=False)
size_kb = PATCH_TARGET_CHECKPOINT.stat().st_size / 1024
print(f"\nsaved: {PATCH_TARGET_CHECKPOINT}  ({size_kb:,.1f} KB)")
print("\ndone — re-run the report cell to see the KB tab populated.")
print(f"if anything looks off, restore with:  cp {backup_path} {PATCH_TARGET_CHECKPOINT}")


backup written:
  checkpoints/evals_czkb_exp_002_baseline_no_expected_enums_high_online_adhoc_quiet_hawk_score.csv.bak.20260515_153826

checkpoint rows:  1123
source rows:      1123
join key:        'trace_id'

source rows with ≥1 non-empty CZ description: 1092 / 1123

rows updated per column:
  reranked_enums_kb_cz       1123 / 1123
  reranked_enums_kb_en       1123 / 1123

saved: checkpoints/evals_czkb_exp_002_baseline_no_expected_enums_high_online_adhoc_quiet_hawk_score.csv  (352,264.5 KB)

done — re-run the report cell to see the KB tab populated.
if anything looks off, restore with:  cp checkpoints/evals_czkb_exp_002_baseline_no_expected_enums_high_online_adhoc_quiet_hawk_score.csv.bak.20260515_153826 checkpoints/evals_czkb_exp_002_baseline_no_expected_enums_high_online_adhoc_quiet_hawk_score.csv
